# Dataset Prep

In [2]:
import shutil
import os
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split

train_dir = '../../data/amia-public-challenge-2026/train/train'

train_csv_path= "../../data/amia-public-challenge-2026/train.csv"
image_size_df= pd.read_csv("../../data/amia-public-challenge-2026/img_size.csv")

RANDOM_SEED= 40
df= pd.read_csv(train_csv_path)

image_ids = df["image_id"].unique()
train_ids, val_ids = train_test_split(
    image_ids,
    test_size=0.2,
    random_state=RANDOM_SEED
)



yolo_dir = f"../../data/yolo_dataset"

def prep_yolo_split(image_ids, split):
    img_dir = os.path.join(yolo_dir, split, 'images')
    lbl_dir = os.path.join(yolo_dir, split, 'labels')
    os.makedirs(img_dir, exist_ok=True); os.makedirs(lbl_dir, exist_ok=True)

    for img_name in tqdm(image_ids, desc=f"YOLO {split}"):
        src = os.path.join(train_dir, img_name + ".png")
        if os.path.exists(src): shutil.copy2(src, os.path.join(img_dir, img_name + ".png"))

        rows = df[df['image_id'] == img_name]
        image_size= image_size_df.loc[image_size_df['image_id'] == img_name]
        x, y= image_size['dim1'], image_size['dim0'] # from kaggle:the original dimensions of the files: dim0 is the height and dim1 is the width.
        original_x, original_y= x.values[0], y.values[0]
        x_scale= 1024/original_x
        y_scale= 1024/original_y
        lines = []
        for _, r in rows.iterrows():
            if r['class_id'] == 14: continue
            xc = ((r['x_min']*x_scale + r['x_max']*x_scale)/2) / 1024
            yc = ((r['y_min']*y_scale + r['y_max']*y_scale)/2) / 1024
            w = (r['x_max']*x_scale - r['x_min']*x_scale) / 1024
            h = (r['y_max']*y_scale - r['y_min']*y_scale) / 1024
            assert 0 <= xc <= 1
            assert 0 <= yc <= 1
            assert 0 <= w <= 1
            assert 0 <= h <= 1
            lines.append(f"{r['class_id']} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        with open(os.path.join(lbl_dir, img_name + ".txt"), 'w') as f:
            f.write('\n'.join(lines))

prep_yolo_split(train_ids, 'train')
prep_yolo_split(val_ids, 'val')




YOLO val: 100%|██████████| 1715/1715 [00:26<00:00, 63.99it/s]


In [3]:
import yaml
yaml_path = os.path.join(yolo_dir, 'dataset.yaml')
CLASS_NAMES = [
        "Aortic enlargement", "Atelectasis", "Calcification", "Cardiomegaly",
        "Consolidation", "ILD", "Infiltration", "Lung Opacity", "Nodule/Mass",
        "Other lesion", "Pleural effusion", "Pleural thickening", 
        "Pneumothorax", "Pulmonary fibrosis", "No finding"
    ]
with open(yaml_path, 'w') as f:
    yaml.dump({
        'path': yolo_dir, 'train': 'train/images', 'val': 'val/images',
        'names': {i: n for i, n in enumerate(CLASS_NAMES[:-1])}
    }, f, default_flow_style=False)



In [ ]:
from ultralytics import YOLO
import torch

yolo_model = YOLO('yolov8m.pt')  # or yolov8x.pt for larger model

yolo_results = yolo_model.train(
    data=yaml_path,
    epochs=10,
    imgsz=1024,
    batch=4,
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=5,
    save=True,
    project="../../outputs/yolo_runs",
    name="chest_xray",
    exist_ok=True,
    mosaic=1.0,      # Enable mosaic augmentation for rare classes
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5, translate=0.1, scale=0.1, shear=2,
    fliplr=0.5,
)